In [ ]:
# Explanation: Marks the start of the section that imports the libraries used by the notebook.
# Modules import
# Explanation: Imports NumPy with the short name np for numerical array operations.
import numpy as np
# Explanation: Imports pandas with the short name pd for working with tabular data.
import pandas as pd
# Explanation: Imports Matplotlib's plotting interface with the short name plt.
import matplotlib.pyplot as plt
# Explanation: Imports Seaborn with the short name sns for statistical visualizations.
import seaborn as sns

In [ ]:
# Explanation: Loads train.csv into a pandas DataFrame named titanic_data.
titanic_data = pd.read_csv('train.csv')

In [ ]:
# Explanation: Displays summary statistics for the numerical columns in the training data.
titanic_data.describe()

In [ ]:
# Explanation: Displays column names, data types, non-null counts, and memory usage.
titanic_data.info()

In [ ]:
# Explanation: Counts the missing values in each column.
titanic_data.isnull().sum()

In [ ]:
# Explanation: Calculates correlations among numerical columns and draws them as a heatmap.
sns.heatmap(titanic_data.corr(numeric_only=True), cmap="YlGnBu")
# Explanation: Renders the plots that have been prepared in the current cell.
plt.show()

In [ ]:
# Explanation: Imports the splitter used to preserve selected group proportions in train and test sets.
from sklearn.model_selection import StratifiedShuffleSplit

# Explanation: Creates one stratified split with 20 percent of the rows assigned to the test set.
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2)
# Explanation: Generates train and test row indices while stratifying by survival, passenger class, and sex.
for train_indices, test_indices in split.split(titanic_data, titanic_data[["Survived", "Pclass", "Sex"]]):
    # Explanation: Selects the rows identified for the stratified training set.
    strat_train_set = titanic_data.loc[train_indices]
    # Explanation: Selects the rows identified for the stratified test set.
    strat_test_set = titanic_data.loc[test_indices]

In [ ]:
# Explanation: Displays the stratified test DataFrame as the cell's output.
strat_test_set 

In [ ]:
# Explanation: Activates the first plot in a one-row, two-column subplot layout.
plt.subplot(1,2,1)
# Explanation: Adds a histogram of survival values from the training set to the current subplot.
strat_train_set['Survived'].hist()
# Explanation: Adds a histogram of passenger-class values from the training set to the current subplot.
strat_train_set['Pclass'].hist()

# Explanation: Activates the second plot in the one-row, two-column subplot layout.
plt.subplot(1,2,2)
# Explanation: Adds a histogram of survival values from the test set to the current subplot.
strat_test_set['Survived'].hist()
# Explanation: Adds a histogram of passenger-class values from the test set to the current subplot.
strat_test_set['Pclass'].hist()

# Explanation: Renders the plots that have been prepared in the current cell.
plt.show()

In [ ]:
# Explanation: Displays the structure and missing-value counts of the stratified training set.
strat_train_set.info()

In [ ]:
# Explanation: Imports scikit-learn base classes used to create pipeline-compatible custom transformers.
from sklearn.base import BaseEstimator, TransformerMixin
# Explanation: Imports SimpleImputer for replacing missing values.
from sklearn.impute import SimpleImputer

# Explanation: Defines a custom transformer that fills missing values in the Age column.
class AgeImputer(BaseEstimator, TransformerMixin):
    # Explanation: Defines the fit method required by scikit-learn transformers; y is optional because this preprocessing is unsupervised.
    def fit(self, X, y=None):
        # Explanation: Returns the fitted transformer instance so it can be used in a scikit-learn pipeline.
        return self

    # Explanation: Defines how the transformer changes the input feature DataFrame X.
    def transform(self, X):
        # Explanation: Creates an imputer that replaces missing values with the column mean.
        imputer = SimpleImputer(strategy="mean")
        # Explanation: Learns the mean Age, fills missing Age values, and assigns the result back to the Age column.
        X["Age"] = imputer.fit_transform(X[["Age"]])
        # Explanation: Returns the transformed DataFrame for the next pipeline step.
        return X

In [ ]:
# Explanation: Imports OneHotEncoder for converting categorical values into numeric indicator columns.
from sklearn.preprocessing import OneHotEncoder

# Explanation: Defines a custom transformer that one-hot encodes Embarked and Sex.
class FeatureEncoder(BaseEstimator, TransformerMixin):
    # Explanation: Defines the fit method required by scikit-learn transformers; y is optional because this preprocessing is unsupervised.
    def fit(self, X, y=None):
        # Explanation: Returns the fitted transformer instance so it can be used in a scikit-learn pipeline.
        return self

    # Explanation: Defines how the transformer changes the input feature DataFrame X.
    def transform(self, X):
        # Explanation: Creates a one-hot encoder with scikit-learn's default settings.
        encoder = OneHotEncoder()
        # Explanation: Learns the Embarked categories and converts their encoded values from a sparse matrix to a dense array.
        matrix = encoder.fit_transform(X[['Embarked']]).toarray()

        # Explanation: Defines names for the expected Embarked indicator columns, including N for a missing category.
        column_names = ["C", "Q", "S", "N"]
        
        # Explanation: Loops over each encoded feature column by using the transposed matrix.
        for i in range(len(matrix.T)):
            # Explanation: Copies the current encoded feature column into the DataFrame under its assigned name.
            X[column_names[i]] = matrix.T[i]

        # Explanation: Refits the encoder on Sex and converts the encoded values to a dense array.
        matrix = encoder.fit_transform(X[['Sex']]).toarray()

        # Explanation: Defines names for the two expected Sex indicator columns.
        column_names = ["Female", "Male"]

        # Explanation: Loops over each encoded feature column by using the transposed matrix.
        for i in range(len(matrix.T)):
            # Explanation: Copies the current encoded feature column into the DataFrame under its assigned name.
            X[column_names[i]] = matrix.T[i]

        # Explanation: Returns the transformed DataFrame for the next pipeline step.
        return X

In [ ]:
# Explanation: Defines a custom transformer that removes columns not used by the model.
class FeatureDropper(BaseEstimator, TransformerMixin):
    # Explanation: Defines the fit method required by scikit-learn transformers; y is optional because this preprocessing is unsupervised.
    def fit(self, X, y=None):
        # Explanation: Returns the fitted transformer instance so it can be used in a scikit-learn pipeline.
        return self

    # Explanation: Defines how the transformer changes the input feature DataFrame X.
    def transform(self, X):
        # Explanation: Returns the DataFrame after dropping the original categorical or text columns and the N indicator, ignoring absent columns.
        return X.drop(columns=["Embarked", "Name", "Ticket", "Cabin", "Sex", "N"], errors="ignore")

In [ ]:
# Explanation: Imports Pipeline so preprocessing steps can be run in a fixed sequence.
from sklearn.pipeline import Pipeline 

# Explanation: Starts a preprocessing pipeline whose first step fills missing Age values.
pipeline = Pipeline([("ageimputer", AgeImputer()),
                     # Explanation: Adds the categorical feature-encoding transformer as the second pipeline step.
                     ("featureencoder", FeatureEncoder()),
                     # Explanation: Adds the column-dropping transformer as the final step and closes the pipeline definition.
                     ("featuredropper", FeatureDropper())])

In [ ]:
# Explanation: Fits the preprocessing pipeline on the training set, transforms it, and replaces the original training DataFrame.
strat_train_set = pipeline.fit_transform(strat_train_set)

In [ ]:
# Explanation: Displays the structure and missing-value counts of the stratified training set.
strat_train_set.info()

In [ ]:
# Explanation: Imports StandardScaler for centering features and scaling them to unit variance.
from sklearn.preprocessing import StandardScaler

# Explanation: Creates the training feature table by removing the Survived target column.
X = strat_train_set.drop(columns=['Survived'])
# Explanation: Creates the training target series from the Survived column.
y = strat_train_set['Survived']

# Explanation: Creates a new standard-scaling transformer.
scaler = StandardScaler()
# Explanation: Learns each training feature's mean and standard deviation, then scales the training features.
X_data = scaler.fit_transform(X)
# Explanation: Converts the pandas target series into a NumPy array.
y_data = y.to_numpy()

In [ ]:
# Explanation: Marks the start of the random-forest model training section.
# Random Forest
# Explanation: Imports the random-forest classifier.
from sklearn.ensemble import RandomForestClassifier
# Explanation: Imports cross-validated grid search for hyperparameter tuning.
from sklearn.model_selection import GridSearchCV

# Explanation: Creates the random-forest classifier that will be tuned.
clf = RandomForestClassifier()

# Explanation: Starts the list of hyperparameter combinations to search; the variable name is kept exactly as written.
param_gird = [
    # Explanation: Starts the dictionary that maps each hyperparameter to its candidate values.
    {
        # Explanation: Provides candidate numbers of decision trees for the forest.
        "n_estimators": [10, 100, 200, 500],
        # Explanation: Provides candidate maximum tree depths, where None allows unrestricted depth.
        "max_depth": [None, 5, 10],
        # Explanation: Provides candidate minimum sample counts required to split an internal tree node.
        "min_samples_split": [2,3,4]
    # Explanation: Closes the hyperparameter dictionary.
    }
# Explanation: Closes the list containing the hyperparameter search dictionary.
]

# Explanation: Configures a three-fold accuracy grid search for clf and asks it to retain training scores.
grid_search = GridSearchCV(clf, param_gird, cv=3, scoring="accuracy", return_train_score=True)
# Explanation: Runs cross-validation for every candidate combination using the scaled training data.
grid_search.fit(X_data, y_data)

In [ ]:
# Explanation: Stores the fitted classifier with the best cross-validation score.
final_clf = grid_search.best_estimator_

In [ ]:
# Explanation: Displays the selected classifier and its chosen hyperparameters.
final_clf

In [ ]:
# Explanation: Fits and applies the preprocessing pipeline to the stratified test set, then replaces that DataFrame.
strat_test_set = pipeline.fit_transform(strat_test_set)

In [ ]:
# Explanation: Creates the test feature table by removing the Survived target column.
X_test = strat_test_set.drop(columns=['Survived'])
# Explanation: Creates the test target series from the Survived column.
y_test = strat_test_set['Survived']

# Explanation: Creates a new standard-scaling transformer.
scaler = StandardScaler()
# Explanation: Fits this new scaler on the test features and transforms those features.
X_data_test = scaler.fit_transform(X_test)
# Explanation: Converts the test target series into a NumPy array.
y_data_test = y_test.to_numpy()

In [ ]:
# Explanation: Calculates and displays the selected classifier's accuracy on the processed test set.
final_clf.score(X_data_test, y_data_test)

In [ ]:
# Explanation: Fits and applies the preprocessing pipeline to all labeled training rows.
final_data = pipeline.fit_transform(titanic_data)

In [ ]:
# Explanation: Displays the fully preprocessed labeled dataset.
final_data

In [ ]:
# Explanation: Creates the full feature table by removing the Survived target column.
X_final = final_data.drop(columns=['Survived'])
# Explanation: Creates the full target series from the Survived column.
y_final = final_data['Survived']

# Explanation: Creates a new standard-scaling transformer.
scaler = StandardScaler()
# Explanation: Learns scaling statistics from all labeled features and transforms those features.
X_data_final = scaler.fit_transform(X_final)
# Explanation: Converts the full target series into a NumPy array.
y_data_final = y_final.to_numpy()

In [ ]:
# Explanation: Creates a fresh random-forest classifier for the final production model.
prod_clf = RandomForestClassifier()

# Explanation: Starts the list of hyperparameter combinations to search; the variable name is kept exactly as written.
param_gird = [
    # Explanation: Starts the dictionary that maps each hyperparameter to its candidate values.
    {
        # Explanation: Provides candidate numbers of decision trees for the forest.
        "n_estimators": [10, 100, 200, 500],
        # Explanation: Provides candidate maximum tree depths, where None allows unrestricted depth.
        "max_depth": [None, 5, 10],
        # Explanation: Provides candidate minimum sample counts required to split an internal tree node.
        "min_samples_split": [2,3,4]
    # Explanation: Closes the hyperparameter dictionary.
    }
# Explanation: Closes the list containing the hyperparameter search dictionary.
]

# Explanation: Configures a three-fold accuracy grid search for the production classifier.
grid_search = GridSearchCV(prod_clf, param_gird, cv=3, scoring="accuracy", return_train_score=True)
# Explanation: Tunes and fits the production classifier using all processed labeled data.
grid_search.fit(X_data_final, y_data_final)

In [ ]:
# Explanation: Stores the fitted production classifier with the best cross-validation score.
prod_final_clf = grid_search.best_estimator_

In [ ]:
# Explanation: Loads the unlabeled Titanic test.csv file into a DataFrame.
titanic_test_data = pd.read_csv('test.csv')

In [ ]:
# Explanation: Fits and applies the preprocessing pipeline to the unlabeled test data.
final_test_data = pipeline.fit_transform(titanic_test_data)

In [ ]:
# Explanation: Assigns the processed unlabeled data to the final prediction feature variable.
X_final_test = final_test_data
# Explanation: Forward-fills remaining missing values using the preceding value in each column.
X_final_test = X_final_test.ffill()

# Explanation: Creates a new standard-scaling transformer.
scaler = StandardScaler()
# Explanation: Fits this new scaler on the unlabeled test features and transforms them.
X_data_final_test = scaler.fit_transform(X_final_test)

In [ ]:
# Explanation: Uses the production classifier to predict survival for every unlabeled test row.
predictions = prod_final_clf.predict(X_data_final_test)

In [ ]:
# Explanation: Displays the array of predicted survival labels.
predictions

In [ ]:
# Explanation: Creates the submission DataFrame with PassengerId values from the original test data.
final_df = pd.DataFrame(titanic_test_data['PassengerId'])
# Explanation: Adds the predicted survival labels as the Survived column.
final_df['Survived'] = predictions
# Explanation: Writes the submission DataFrame to predictions.csv without an extra index column.
final_df.to_csv('predictions.csv', index=False)

In [ ]:
# Explanation: Displays the completed submission DataFrame.
final_df